# Quantitative experiment: Online STDR time and memory

This notebook runs `OnlineSTDRMiner` inline for multiple event-type-count scenarios and records per-event execution time and process memory usage. By default it tests the top `(25, 50, 75, 100)` event types, uses the entire selected stream for each scenario, and uses `theta=0.3`.

For each scenario, event types are selected by frequency in the full dataset. All per-event measurements are saved into one combined CSV so the final plot can be recreated later.

In [1]:
from __future__ import annotations

from datetime import datetime
from math import log
from pathlib import Path
import json
import os
import sys
from time import perf_counter

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "Notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

os.environ.setdefault("MPLCONFIGDIR", str(PROJECT_ROOT / ".matplotlib_cache"))

from IPython.display import DisplayHandle, Markdown
import matplotlib.pyplot as plt
import pandas as pd
import psutil

sys.path.insert(0, str(PROJECT_ROOT / "Source"))

from spatio_temporal_network import (
    LearningParameters,
    OnlineSTDRMiner,
    initialize_network_from_events,
)

pd.set_option("display.max_columns", 100)

In [ ]:
# Scenario controls. None would mean all event types; default scenarios use top 25, 50, 75, and 100.
EVENT_TYPE_COUNTS: tuple[int | None, ...] = (25, 50, 75, 100)
MAX_EVENTS: int | None = None

DATA_PATH = PROJECT_ROOT / "Data" / "Preprocessed" / "dataset_TSMC2014_NYC_preprocessed.csv"
OUTPUT_ROOT = PROJECT_ROOT / "Results" / "Quantitative_experiment_time_memory"
RUN_ID = datetime.now().strftime("run_%Y%m%d_%H%M%S")
RUN_DIR = OUTPUT_ROOT / RUN_ID

# Parameters copied from Notebooks/quantitative_experiment_comparison.ipynb.
GRID_SHAPE = (10, 10)
ONLINE_SPATIAL_THRESHOLD_METERS = 10_000.0
ONLINE_ETA = 1.0
ONLINE_TAU_ACTIVATION = 60.0
ONLINE_TAU_REFRACTORY = 12
ONLINE_LAMBDA_DECAY = -log(0.10) / 120.0
MAX_RULE_LENGTH = 2
MAX_SNAPSHOT_RULES = None

# One theta is tested initially. Change this tuple to compare more theta values.
ONLINE_THETAS_TO_RUN = (0.15,)

# The notebook updates one live status display at this interval.
PROGRESS_EVERY = 100

# Plot colors follow the quantitative comparison notebook palette.
SCENARIO_COLORS = {
    25: "#4D4D4D",
    50: "#E69F00",
    75: "#009E73",
    100: "#56B4E9",
    None: "#666666",
}

RUN_DIR.mkdir(parents=True, exist_ok=True)
RUN_DIR

PosixPath('/Users/piotr/Praca/Nauka/publikacje/SNNs_ST_Patterns/Experiments_software/Results/Quantitative_experiment_time_memory/run_20260604_195114')

In [3]:
def event_type_count_label(event_type_count: int | None) -> str:
    return "all" if event_type_count is None else f"{event_type_count:03d}"


def scenario_label(event_type_count: int | None) -> str:
    return "all event types" if event_type_count is None else f"{event_type_count} event types"


def load_selected_events(
    raw_events: pd.DataFrame,
    event_type_count: int | None = None,
    max_events: int | None = None,
) -> tuple[pd.DataFrame, tuple[str, ...]]:
    ranked_types = tuple(raw_events["Event_type"].value_counts().index)
    selected_types = ranked_types if event_type_count is None else ranked_types[:event_type_count]
    selected_events = raw_events[raw_events["Event_type"].isin(selected_types)].copy()
    selected_events = selected_events.sort_values("timestamp", ignore_index=True)
    if max_events is not None:
        selected_events = selected_events.head(max_events).copy()
    if selected_events.empty:
        raise ValueError("No events were selected. Check EVENT_TYPE_COUNTS and MAX_EVENTS.")
    selected_types_used = tuple(
        event_type
        for event_type in selected_types
        if event_type in set(selected_events["Event_type"])
    )
    return selected_events.reset_index(drop=True), selected_types_used


def make_online_parameters(events: pd.DataFrame, online_theta: float) -> LearningParameters:
    observation_window_minutes = float(events["timestamp"].iloc[-1] - events["timestamp"].iloc[0])
    if observation_window_minutes <= 0:
        raise ValueError("The selected stream must cover a positive time span.")
    return LearningParameters(
        eta=ONLINE_ETA,
        lambda_decay=ONLINE_LAMBDA_DECAY,
        tau_activation=ONLINE_TAU_ACTIVATION,
        tau_refractory=ONLINE_TAU_REFRACTORY,
        theta=online_theta,
    )


def merged_event_type_rule_count(rules: list) -> int:
    return len({tuple(rule.event_type_rule) for rule in rules})


def rss_mb(process: psutil.Process) -> float:
    return process.memory_info().rss / (1024 ** 2)


def relative_path(path: Path) -> str:
    return str(path.relative_to(PROJECT_ROOT))

In [4]:
raw_events = pd.read_csv(DATA_PATH).sort_values("timestamp", ignore_index=True)

config = {
    "run_id": RUN_ID,
    "data_path": relative_path(DATA_PATH),
    "event_type_counts_requested": list(EVENT_TYPE_COUNTS),
    "max_events_requested": MAX_EVENTS,
    "online_thetas_to_run": list(ONLINE_THETAS_TO_RUN),
    "grid_shape": GRID_SHAPE,
    "online_spatial_threshold_meters": ONLINE_SPATIAL_THRESHOLD_METERS,
    "online_eta": ONLINE_ETA,
    "online_tau_activation": ONLINE_TAU_ACTIVATION,
    "online_tau_refractory": ONLINE_TAU_REFRACTORY,
    "online_lambda_decay": ONLINE_LAMBDA_DECAY,
    "max_rule_length": MAX_RULE_LENGTH,
    "max_snapshot_rules": MAX_SNAPSHOT_RULES,
    "progress_every": PROGRESS_EVERY,
}
(RUN_DIR / "experiment_config.json").write_text(json.dumps(config, indent=2), encoding="utf-8")

print(f"Loaded {len(raw_events):,} total events and {raw_events['Event_type'].nunique():,} event types.")
print(f"Scenarios: {EVENT_TYPE_COUNTS}")
print(f"Results directory: {RUN_DIR}")

Loaded 227,428 total events and 251 event types.
Scenarios: (None,)
Results directory: /Users/piotr/Praca/Nauka/publikacje/SNNs_ST_Patterns/Experiments_software/Results/Quantitative_experiment_time_memory/run_20260604_195114


In [5]:
def run_online_time_memory(
    events: pd.DataFrame,
    selected_event_types: tuple[str, ...],
    requested_event_type_count: int | None,
    online_theta: float,
    output_dir: Path,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    output_dir.mkdir(parents=True, exist_ok=True)
    process = psutil.Process()
    status = DisplayHandle()
    status.display(Markdown("Starting Online STDR run..."))

    setup_start = perf_counter()
    setup_memory_before_mb = rss_mb(process)
    parameters = make_online_parameters(events, online_theta)
    network = initialize_network_from_events(
        events,
        grid_shape=GRID_SHAPE,
        spatial_threshold=ONLINE_SPATIAL_THRESHOLD_METERS,
    )
    miner = OnlineSTDRMiner(network, parameters)
    setup_seconds = perf_counter() - setup_start
    setup_memory_after_mb = rss_mb(process)

    rows = []
    cumulative_event_seconds = 0.0
    cumulative_process_seconds = 0.0
    cumulative_extract_seconds = 0.0
    peak_rss_mb = setup_memory_after_mb
    total_events = len(events)
    label = scenario_label(requested_event_type_count)

    for event_index, event in enumerate(events.itertuples(index=False), start=1):
        event_series = pd.Series(event._asdict())

        event_start = perf_counter()
        process_start = perf_counter()
        miner.process_event(event_series, extract_rules=False)
        process_seconds = perf_counter() - process_start

        extract_start = perf_counter()
        rules = miner.extract_rules(
            max_rule_length=MAX_RULE_LENGTH,
            max_rules=MAX_SNAPSHOT_RULES,
        )
        extract_seconds = perf_counter() - extract_start
        event_seconds = perf_counter() - event_start

        current_rss_mb = rss_mb(process)
        peak_rss_mb = max(peak_rss_mb, current_rss_mb)
        cumulative_event_seconds += event_seconds
        cumulative_process_seconds += process_seconds
        cumulative_extract_seconds += extract_seconds

        row = {
            "run_id": RUN_ID,
            "algorithm": "online_stdr",
            "online_theta": online_theta,
            "requested_event_type_count": requested_event_type_count,
            "event_type_count": len(selected_event_types),
            "scenario_label": label,
            "event_index": event_index,
            "event_id": getattr(event, "Event_ID"),
            "event_type": getattr(event, "Event_type"),
            "event_timestamp": float(getattr(event, "timestamp")),
            "event_seconds": event_seconds,
            "process_event_seconds": process_seconds,
            "extract_rules_seconds": extract_seconds,
            "cumulative_event_seconds": cumulative_event_seconds,
            "cumulative_process_event_seconds": cumulative_process_seconds,
            "cumulative_extract_rules_seconds": cumulative_extract_seconds,
            "rss_mb": current_rss_mb,
            "rss_delta_from_setup_start_mb": current_rss_mb - setup_memory_before_mb,
            "peak_rss_mb": peak_rss_mb,
            "rule_count": len(rules),
            "merged_event_type_rule_count": merged_event_type_rule_count(rules),
            "significant_synapse_count": len(miner.significant_synapses),
            "stored_synapse_state_count": len(miner.network.synapse_states),
            "max_events_requested": MAX_EVENTS,
        }
        rows.append(row)

        if event_index % PROGRESS_EVERY == 0 or event_index == total_events:
            status.update(Markdown(
                "\n".join([
                    f"**Online STDR running: {label}, theta={online_theta:g}**",
                    f"Events: {event_index:,} / {total_events:,}",
                    f"Latest event time: {event_seconds:.6f} s",
                    f"Cumulative execution time: {cumulative_event_seconds:.3f} s",
                    f"RSS memory: {current_rss_mb:.2f} MB",
                    f"Peak RSS memory: {peak_rss_mb:.2f} MB",
                    f"Rules: {len(rules):,}; significant synapses: {len(miner.significant_synapses):,}",
                ])
            ))

    measurements = pd.DataFrame(rows)
    final_rules = miner.extract_rules(max_rule_length=MAX_RULE_LENGTH, max_rules=MAX_SNAPSHOT_RULES)
    final_rules_frame = miner.rules_frame(final_rules)

    measurements.to_csv(output_dir / "time_memory_measurements.csv", index=False)
    measurements.to_pickle(output_dir / "time_memory_measurements.pkl")
    final_rules_frame.to_csv(output_dir / "final_rules.csv", index=False)
    miner.significant_synapses_frame().to_csv(output_dir / "final_significant_synapses.csv", index=False)

    metadata = {
        **config,
        "online_theta": online_theta,
        "requested_event_type_count": requested_event_type_count,
        "event_type_count_used": len(selected_event_types),
        "event_count_used": len(events),
        "network": network.summary(),
        "setup_seconds": setup_seconds,
        "setup_memory_before_mb": setup_memory_before_mb,
        "setup_memory_after_mb": setup_memory_after_mb,
        "setup_memory_delta_mb": setup_memory_after_mb - setup_memory_before_mb,
        "final_cumulative_event_seconds": cumulative_event_seconds,
        "final_rss_mb": rss_mb(process),
        "final_peak_rss_mb": peak_rss_mb,
        "final_rule_count": len(final_rules),
        "final_merged_event_type_rule_count": merged_event_type_rule_count(final_rules),
        "final_significant_synapse_count": len(miner.significant_synapses),
    }
    (output_dir / "run_metadata.json").write_text(json.dumps(metadata, indent=2), encoding="utf-8")
    return measurements, final_rules_frame

In [ ]:
experiment_start = perf_counter()
all_measurements = []
final_rules_by_scenario = {}
selected_type_rows = []

for requested_event_type_count in EVENT_TYPE_COUNTS:
    selected_events, selected_types = load_selected_events(
        raw_events,
        requested_event_type_count,
        MAX_EVENTS,
    )
    scenario_dir = RUN_DIR / f"event_types_{event_type_count_label(requested_event_type_count)}"
    selection_dir = scenario_dir / "selection"
    selection_dir.mkdir(parents=True, exist_ok=True)
    selected_events.to_csv(selection_dir / "selected_events.csv", index=False)
    pd.DataFrame(
        {"rank": range(1, len(selected_types) + 1), "event_type": selected_types}
    ).to_csv(selection_dir / "selected_event_types.csv", index=False)
    selected_type_rows.extend(
        {
            "requested_event_type_count": requested_event_type_count,
            "event_type_count": len(selected_types),
            "rank": rank,
            "event_type": event_type,
        }
        for rank, event_type in enumerate(selected_types, start=1)
    )

    print(
        f"Running {scenario_label(requested_event_type_count)}: "
        f"{len(selected_events):,} events, {len(selected_types):,} selected event types"
    )
    for online_theta in ONLINE_THETAS_TO_RUN:
        theta_label = f"theta_{online_theta:.2f}".replace(".", "_")
        theta_dir = scenario_dir / theta_label
        measurements, final_rules = run_online_time_memory(
            selected_events,
            selected_types,
            requested_event_type_count,
            online_theta,
            theta_dir,
        )
        all_measurements.append(measurements)
        final_rules_by_scenario[(requested_event_type_count, online_theta)] = final_rules

measurements_df = pd.concat(all_measurements, ignore_index=True)
selected_event_types_df = pd.DataFrame(selected_type_rows)
measurements_df.to_csv(RUN_DIR / "time_memory_measurements.csv", index=False)
measurements_df.to_pickle(RUN_DIR / "time_memory_measurements.pkl")
selected_event_types_df.to_csv(RUN_DIR / "selected_event_types.csv", index=False)

elapsed_seconds = perf_counter() - experiment_start
pd.DataFrame([{"run_id": RUN_ID, "elapsed_seconds": elapsed_seconds}]).to_csv(
    RUN_DIR / "run_metadata.csv",
    index=False,
)

measurements_df.head()

Running all event types: 227,428 events, 251 selected event types


Starting Online STDR run...

In [ ]:
summary_df = (
    measurements_df
    .sort_values(["requested_event_type_count", "online_theta", "event_index"])
    .groupby(["requested_event_type_count", "online_theta"], as_index=False)
    .tail(1)[[
        "requested_event_type_count",
        "event_type_count",
        "scenario_label",
        "online_theta",
        "event_index",
        "cumulative_event_seconds",
        "cumulative_process_event_seconds",
        "cumulative_extract_rules_seconds",
        "rss_mb",
        "peak_rss_mb",
        "rule_count",
        "merged_event_type_rule_count",
        "significant_synapse_count",
        "stored_synapse_state_count",
    ]]
    .sort_values(["requested_event_type_count", "online_theta"], ignore_index=True)
)
summary_df.to_csv(RUN_DIR / "summary.csv", index=False)
summary_df

## Recreate Plot From CSV

The combined plot can be recreated from `time_memory_measurements.csv` without rerunning the miner.

In [1]:
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "Notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

OUTPUT_ROOT = PROJECT_ROOT / "Results" / "Quantitative_experiment_time_memory"
if "RUN_DIR" not in globals() or not Path(RUN_DIR).exists():
    run_dirs = sorted(
        path for path in OUTPUT_ROOT.iterdir()
        if path.is_dir() and path.name.startswith("run_")
    )
    if not run_dirs:
        raise FileNotFoundError(f"No run_* directories found in {OUTPUT_ROOT}")
    RUN_DIR = run_dirs[-1]
else:
    RUN_DIR = Path(RUN_DIR)

reloaded_measurements = pd.read_csv(RUN_DIR / "time_memory_measurements.csv")
print(f"Using run directory: {RUN_DIR}")
reloaded_measurements.head()

Using run directory: /Users/piotr/Praca/Nauka/publikacje/SNNs_ST_Patterns/Experiments_software/Results/Quantitative_experiment_time_memory/run_20260604_200025


,run_id,algorithm,online_theta,requested_event_type_count,event_type_count,scenario_label,event_index,event_id,event_type,event_timestamp,...,cumulative_process_event_seconds,cumulative_extract_rules_seconds,rss_mb,rss_delta_from_setup_start_mb,peak_rss_mb,rule_count,merged_event_type_rule_count,significant_synapse_count,stored_synapse_state_count,max_events_requested
0,run_20260604_200025,online_stdr,0.15,25,25,25 event types,1,3,Home (private),2.250000,...,0.005906,0.000008,283.671875,62.703125,283.671875,0,0,0,624,NaN
1,run_20260604_200025,online_stdr,0.15,25,25,25 event types,2,4,Medical Center,2.533333,...,0.011741,0.000014,283.671875,62.703125,283.671875,0,0,0,1248,NaN
2,run_20260604_200025,online_stdr,0.15,25,25,25 event types,3,6,Food & Drink Shop,3.850000,...,0.017939,0.000022,283.687500,62.718750,283.687500,0,0,0,1868,NaN
3,run_20260604_200025,online_stdr,0.15,25,25,25 event types,4,7,Coffee Shop,4.483333,...,0.023640,0.000027,283.859375,62.890625,283.859375,0,0,0,2486,NaN
4,run_20260604_200025,online_stdr,0.15,25,25,25 event types,5,8,Bus Station,4.550000,...,0.029889,0.000035,284.046875,63.078125,284.046875,0,0,0,3104,NaN
